In [0]:
dbutils.fs.ls("/Volumes/adb_caresync/landing/landing-volume")

In [0]:
dbutils.widgets.text("source_system","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_file_format","")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Define paths
base_landing_path = "/Volumes/adb_caresync/landing/landing-volume/"
source_system_path = f"{base_landing_path}/{source_system}/"
table_path = f"{source_system_path}/{table_name}/"
landing_path = f"{table_path}/{landing_timestamp}/"

print(f"Base landing path: {base_landing_path}")
print(f"Source system path: {source_system_path}")
print(f"Table path: {table_path}")
print(f"Landing path: {landing_path}")


def path_exists(path):
    """Check if a given path exists in DBFS/Volumes."""
    try:
        dbutils.fs.ls(path)
        print(f"Path exists: {path}")
        return True
    except Exception:
        print(f"Path does NOT exist: {path}")
        return False


# Check each path in order; stop at the first missing one
missing_path = next(
    (path for path in [base_landing_path, source_system_path, table_path, landing_path] if not path_exists(path)),
    None,
)

if missing_path:
    message = (
        f"No new files are received for source_system={source_system}, "
        f"table_name={table_name}, landing_timestamp={landing_timestamp}. "
        f"Missing path: {missing_path}"
    )
    print(message)
    dbutils.notebook.exit(message)
else:
    print("All paths verified successfully. Proceeding...")

In [0]:
df = spark.read.format("parquet").load(f"/Volumes/adb_caresync/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}")

if source_file_format=='csv':
    df = spark.read.format("csv").option("header",True).option("inferSchema",True).load(f"/Volumes/adb_caresync/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}")
else:
    df = spark.read.format("parquet").load(f"/Volumes/adb_caresync/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}")


In [0]:
from pyspark.sql import functions as F

updated_df = df.withColumn("landing_timestamp",F.lit(landing_timestamp)).withColumn("insert_timestamp", F.current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")

In [0]:
%sql
select * from adb_caresync.bronze.hospitals;